# 第54章 分面图（FacetGrid）

使用FacetGrid、catplot和relplot把类别映射为可比较的小图。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

一个图过于拥挤，需要按行列分组重复相同图形。

## 数据结构

长表，包含X、Y以及一至两个分面分类变量。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 col_wrap=3 改为 col_wrap=2，观察分面换行对布局的影响
2. 修改 sharex=True 为 sharex=False，对比共享与独立X轴对子图比较的作用
3. 调整 aspect 参数（如 0.8 或 1.2），说明子图宽高比对可读性的影响


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="notebook")
from js import window
base_url = window.location.origin
diamonds = pd.read_csv(f"{base_url}/datasets/diamonds.csv")
orders_full = diamonds.assign(
    category=diamonds["cut"], channel=diamonds["color"], region=diamonds["clarity"],
    order_value=diamonds["price"], items=diamonds["carat"],
    satisfied=np.where(diamonds["price"] >= diamonds["price"].median(), "高于中位价", "不高于中位价"),
)
orders = orders_full.sample(2_000, random_state=36).copy()
taxis = pd.read_csv(f"{base_url}/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"), visits=taxis["distance"],
    ad_spend=taxis["tip"], sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(min(2_000, len(marketing_full)), random_state=36).copy()
flights = pd.read_csv(f"{base_url}/datasets/flights.csv")
daily = flights.assign(
    date=pd.to_datetime(flights["year"].astype("string") + "-" + flights["month"] + "-01"),
    region="AirPassengers", sales=flights["passengers"],
)
print(f"Diamonds：{len(diamonds):,} 行；NYC Taxis：{len(taxis):,} 行；Flights：{len(flights):,} 行")
print("图表兼容列均由公开数据原始字段直接映射；高成本图使用固定 2,000 行样本")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
grid = sns.relplot(data=marketing, x="visits", y="sales", col="channel", col_wrap=3, hue="channel", height=3.3, aspect=1, palette="colorblind", legend=False)
grid.set_axis_labels("访问量", "销售额")
grid.set_titles("{col_name}")
grid.fig.suptitle("分渠道访问量与销售额", y=1.04)
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
grid = sns.catplot(data=orders, x="category", y="order_value", col="region", kind="box", hue="category", palette="Set2", legend=False, height=3.5, aspect=0.9)
grid.set_axis_labels("品类", "客单价（元）")
grid.set_titles("{col_name}")
grid.fig.suptitle("分区域品类客单价", y=1.04)
plt.show()


## 3. 参数说明

- row/col：分面
- col_wrap：换行
- sharex/sharey：共享轴
- height/aspect：尺寸


## 4. 结果解读

在共享坐标下比较模式、斜率和分布；同时检查每个面板样本量。


## 常见误区

- 面板过多
- 坐标不共享却直接比较高低
- 分面和hue重复编码同一变量


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
facet = sns.FacetGrid(daily, row="region", height=2.2, aspect=3, sharex=True, sharey=True, margin_titles=True)
facet.map_dataframe(sns.lineplot, x="date", y="sales", marker="o", errorbar=None, color="#1a73e8")
facet.set_axis_labels("日期", "销售额")
facet.set_titles(row_template="{row_name}")
facet.fig.subplots_adjust(top=0.9)
facet.fig.suptitle("区域每日销售趋势")
plt.show()


## 本章小结

使用FacetGrid、catplot和relplot把类别映射为可比较的小图。


### 你已经掌握

- 判断分面图（FacetGrid）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 一个图过于拥挤，需要按行列分组重复相同图形。 |
| 数据结构 | 长表，包含X、Y以及一至两个分面分类变量。 |
| 结果解读 | 在共享坐标下比较模式、斜率和分布；同时检查每个面板样本量。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `row/col` | 分面 |
| `col_wrap` | 换行 |
| `sharex/sharey` | 共享轴 |
| `height/aspect` | 尺寸 |


### 需要注意

- 面板过多
- 坐标不共享却直接比较高低
- 分面和hue重复编码同一变量


### 完成检查

- [ ] 能判断什么问题适合使用分面图（FacetGrid）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
